In [1]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import ast
import numpy as np
import pandas as pd

# Load climate dataframe
df = pd.read_csv("../../MultimodalForcast/data/climate_ttc/climate_2014_2023_final_with_embeddings_lag_3.csv")

# Sort by date if available
if "date" in df.columns:
    df = df.sort_values("date").reset_index(drop=True)

# Configuration: Select which columns to use as features
# Option 1: Specify columns explicitly
feature_columns = ["precip_lag1", "precip_lag2", "precip_lag3",
                   "humidity_lag1", "humidity_lag2", "humidity_lag3",
                   "windspeed_lag1", "windspeed_lag2", "windspeed_lag3",
                   "temp_lag1", "temp_lag2", "temp_lag3"]

# Option 2: Auto-select all numerical lag columns (uncomment to use)
# exclude_cols = ["date", "temp", "text"] + [col for col in df.columns if "text_lag" in col or "embedding" in col]
# feature_columns = [col for col in df.columns if col not in exclude_cols]

# Validate and extract features
missing_cols = [col for col in feature_columns if col not in df.columns]
if missing_cols:
    print(f"Warning: The following columns are not found in the dataframe: {missing_cols}")
    feature_columns = [col for col in feature_columns if col in df.columns]

X = df[feature_columns].values
y = df["temp"].values

print(f"Using {len(feature_columns)} feature columns: {feature_columns}")

# Function to extract and stack embedding columns
def extract_and_stack_embeddings(df, embedding_cols):
    """
    Extract embedding columns from dataframe and stack them as a numpy matrix.
    
    Args:
        df: DataFrame with embedding columns stored as string representations of lists
        embedding_cols: List of column names to extract (e.g., ['embedding_text_lag1', 'embedding_text_lag2', 'embedding_text_lag3'])
    
    Returns:
        numpy array of shape (N, L, D) where:
        - N = number of rows
        - L = number of embedding columns (lags)
        - D = embedding dimension
    """
    embeddings_list = []
    
    for col in embedding_cols:
        if col not in df.columns:
            print(f"Warning: Column '{col}' not found. Skipping.")
            continue
        
        # Parse string representation of list to actual list
        col_embeddings = []
        for idx, val in enumerate(df[col]):
            try:
                # Try to parse as literal string representation
                if isinstance(val, str):
                    embedding = ast.literal_eval(val)
                else:
                    embedding = val
                col_embeddings.append(np.array(embedding))
            except (ValueError, SyntaxError) as e:
                print(f"Warning: Could not parse embedding at row {idx} in column {col}: {e}")
                # Use zeros as fallback (you may want to handle this differently)
                if len(col_embeddings) > 0:
                    col_embeddings.append(np.zeros_like(col_embeddings[0]))
                else:
                    # If we don't know the dimension yet, skip this row
                    continue
        
        embeddings_list.append(np.array(col_embeddings))
    
    # Stack along the lag dimension: (N, L, D)
    if embeddings_list:
        stacked = np.stack(embeddings_list, axis=1)  # (N, L, D)
        return stacked
    else:
        raise ValueError("No valid embedding columns found")

# Extract text embeddings for all data first
embedding_cols = ['embedding_text_lag1', 'embedding_text_lag2', 'embedding_text_lag3']
text_embeddings_all = extract_and_stack_embeddings(df, embedding_cols)

# Split based on date (time series split)
# Use 70% of data for training (earlier dates), 30% for testing (later dates)
split_ratio = 0.7
split_idx = int(len(df) * split_ratio)

# Alternatively, you can use a specific date:
# split_date = "2022-12-05"
# if "date" in df.columns:
#     df['date'] = pd.to_datetime(df['date'])
#     split_idx = len(df[df['date'] < pd.to_datetime(split_date)])

idx_train = np.arange(split_idx)
idx_test = np.arange(split_idx, len(df))

# Split data using date-based indices
X_train = X[idx_train]
X_test = X[idx_test]
y_train = y[idx_train]
y_test = y[idx_test]

# Split text embeddings using the same indices
train_text = text_embeddings_all[idx_train]
test_text = text_embeddings_all[idx_test]

# Print split information
if "date" in df.columns:
    print(f"Train date range: {df.iloc[idx_train[0]]['date']} to {df.iloc[idx_train[-1]]['date']}")
    print(f"Test date range: {df.iloc[idx_test[0]]['date']} to {df.iloc[idx_test[-1]]['date']}")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"train_text shape: {train_text.shape}")
print(f"test_text shape: {test_text.shape}")


Using 12 feature columns: ['precip_lag1', 'precip_lag2', 'precip_lag3', 'humidity_lag1', 'humidity_lag2', 'humidity_lag3', 'windspeed_lag1', 'windspeed_lag2', 'windspeed_lag3', 'temp_lag1', 'temp_lag2', 'temp_lag3']
Train date range: 2014-01-01 to 2020-12-09
Test date range: 2020-12-10 to 2023-12-01
X_train shape: (2535, 12)
X_test shape: (1087, 12)
y_train shape: (2535,)
y_test shape: (1087,)
train_text shape: (2535, 3, 1024)
test_text shape: (1087, 3, 1024)


In [2]:
from tabdpt import TabDPTRegressor
model = TabDPTRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test, n_ensembles=1, context_size=len(X_train), seed=42)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

RMSE: 4.4126
MAE: 3.3174


In [4]:
from tabdpt import TabDPTRegressor
model = TabDPTRegressor(text_enhanced=True)
model.fit(X_train, y_train, train_text)
y_pred = model.predict(X_test, n_ensembles=1, context_size=len(X_train), seed=42, text=test_text)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")


/home/yuyan/TabDPT-inference-text-enhanced/src/tabdpt/estimator.py:192: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_tensor = torch.tensor(train_text, dtype=torch.float32, device=self.device)  # (N_train, L, D)
/home/yuyan/TabDPT-inference-text-enhanced/src/tabdpt/estimator.py:193: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_tensor = torch.tensor(text_test, dtype=torch.float32, device=self.device)  # (N_test, L, D)


attention weight shape: torch.Size([1, 8, 3622, 2535]), text_enhanced_attn_weight shape: torch.Size([1, 8, 3622, 2535])
attention weight before blending: tensor([[[[3.6754e-04, 2.7512e-05, 5.1519e-06,  ..., 2.6648e-05,
           2.3400e-05, 2.5648e-05],
          [7.2693e-05, 2.4626e-03, 1.9695e-04,  ..., 5.1275e-04,
           9.7036e-04, 5.6784e-04],
          [8.3810e-05, 1.7513e-03, 2.4319e-04,  ..., 5.0482e-04,
           8.4632e-04, 7.2618e-04],
          ...,
          [2.8505e-05, 1.2604e-03, 2.1397e-04,  ..., 4.7192e-04,
           1.0408e-03, 1.1977e-03],
          [4.3887e-05, 1.3619e-03, 2.4177e-04,  ..., 4.8457e-04,
           1.2298e-03, 1.0286e-03],
          [9.6075e-05, 4.0038e-04, 1.0776e-04,  ..., 2.3132e-04,
           4.6661e-04, 5.3010e-04]],

         [[9.2013e-04, 9.8348e-04, 1.3627e-03,  ..., 4.6220e-04,
           4.1206e-04, 3.6972e-04],
          [2.7872e-04, 2.5741e-03, 3.4081e-03,  ..., 5.5376e-04,
           5.7859e-04, 5.5024e-04],
          [3.0008e-04

In [4]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tabdpt import TabDPTRegressor
import numpy as np

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import ast
import pandas as pd


X, y = fetch_california_housing(return_X_y=True)
X, y = X[:4096], y[:4096]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
model = TabDPTRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test, n_ensembles=2, context_size=2048, seed=42)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
print(r2_score(y_test, y_pred))

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

2it [00:15,  7.93s/it]

0.883458479032966
RMSE: 0.3360
MAE: 0.2160
